# H3SFCA 민감도 분석
- 목적: 거리감쇠 방식과 문화누리대상자 수요가중치 설정에 따라 접근성 결과가 얼마나 달라지는지 검토
- 분석 단위: 서울 100m 격자 × 문화시설 중분류
- 분석 대상: 문화누리대상자 추정 인구수가 1명 이상인 격자
- 제외 분류: 음악, 체육용품은 선호 미반영 SFCA 고정값으로 대체된 분류이므로 민감도 비교에서 제외
- 저장 방식: 대시보드·보고서 검토에 필요한 요약 산출물만 저장

In [ ]:
import pandas as pd
import numpy as np
import pathlib
from pathlib import Path

BASE_PATH = pathlib.Path().resolve()

if BASE_PATH.name == "access":
    BASE_PATH = BASE_PATH
elif BASE_PATH.name == "notebooks":
    BASE_PATH = BASE_PATH / "access"
elif (BASE_PATH / "notebooks" / "access").exists():
    BASE_PATH = BASE_PATH / "notebooks" / "access"

OUTPUT_PATH = BASE_PATH / "OUTPUT"
H3_PATH = OUTPUT_PATH / "h3sfca"
SENSITIVITY_PATH = OUTPUT_PATH / "h3sfca_sensitivity"
SENSITIVITY_PATH.mkdir(parents=True, exist_ok=True)

H3_ACCESS_PATH = H3_PATH / "h3sfca_preference_sensitivity_격자_중분류_접근성.parquet"

print("BASE_PATH:", BASE_PATH)
print("H3 접근성 테이블:", H3_ACCESS_PATH)
print("민감도 산출물 폴더:", SENSITIVITY_PATH)

## 1. 분석 설계
- 고정 조건: 시설자료, 공급량, 격자 인구, 문화누리대상자 추정 인구, 생활권 기준, 접근 가능 pair, 시설 중분류
- 변화 조건: 거리감쇠 방식 3개 × 문화누리대상자 수요가중치 3개
- 거리감쇠 방식: `piecewise` = 구간형 감쇠, `gaussian` = 가우시안 연속 감쇠, `none` = 감쇠 없음

\[
	ext{scenario} = \{piecewise, gaussian, none\} 	imes \{1.0, 1.2, 1.5\}
\]

- 접근성 취약지역: 중분류·시나리오별 접근성 하위 20% 격자
- 동점 처리: 접근성지수, GRID_CD 순서로 정렬하여 동일 기준으로 고정

In [ ]:
DECAY_LABEL = {
    "piecewise": "구간형 감쇠",
    "gaussian": "가우시안 감쇠",
    "none": "감쇠 없음"
}

DECAY_ORDER = ["piecewise", "gaussian", "none"]
WEIGHT_ORDER = [1.0, 1.2, 1.5]
EXCLUDED_CATEGORIES = ["음악", "체육용품"]

print("거리감쇠 방식:", DECAY_LABEL)
print("수요가중치 후보:", WEIGHT_ORDER)
print("민감도 비교 제외 분류:", EXCLUDED_CATEGORIES)

## 2. 데이터 불러오기 및 품질 점검
- H3SFCA 9개 시나리오 결과를 불러옴
- 거리감쇠 라벨은 `piecewise`, `gaussian`, `none`으로 통일
- 결측, 음수 접근성, 시나리오 개수를 점검

In [ ]:
h3 = pd.read_parquet(H3_ACCESS_PATH)

h3["문화누리대상자_수요가중치"] = h3["문화누리대상자_수요가중치"].astype(float)
h3["scenario"] = (
    h3["거리감쇠방식"]
    + "_lambda_"
    + h3["문화누리대상자_수요가중치"].map(lambda x: f"{x:.1f}")
)

print("데이터 구조:", h3.shape)
print("중분류 수:", h3["중분류"].nunique())
print("시나리오 수:", h3["scenario"].nunique())
print("거리감쇠 방식:", sorted(h3["거리감쇠방식"].dropna().unique()))
print("수요가중치:", sorted(h3["문화누리대상자_수요가중치"].dropna().unique()))
print("접근성지수 결측:", h3["접근성지수"].isna().sum())
print("접근성지수 음수:", (h3["접근성지수"] < 0).sum())
print("문화누리대상자 추정 인구 음수:", (h3["문화누리대상자_추정_인구수"] < 0).sum())

display(h3.head())

## 3. 분석 대상 격자 설정
- 문화누리대상자 추정 인구가 있는 격자만 민감도 분석 대상으로 사용
- 음악·체육용품은 고정 대체값이므로 제외
- 접근성 취약지역 판단에서도 동일 기준을 적용
- 수요가 없는 격자는 접근성 지표 해석 대상에서 제외

In [ ]:
h3_target = h3[
    (h3["문화누리대상자_추정_인구수"] > 0)
    & (~h3["중분류"].isin(EXCLUDED_CATEGORIES))
].copy()

print("전체 레코드 수:", len(h3))
print("분석 대상 레코드 수:", len(h3_target))
print("분석 대상 GRID_CD 수:", h3_target["GRID_CD"].nunique())
print("분석 대상 시군구 수:", h3_target["시군구"].nunique())
print("분석 대상 행정동 수:", h3_target["행정동"].nunique())
print("분석 대상 중분류:", sorted(h3_target["중분류"].unique()))
print("제외된 중분류:", EXCLUDED_CATEGORIES)

scenario_check = (
    h3_target
    .groupby(["중분류", "scenario"], as_index=False)
    .agg(격자수=("GRID_CD", "nunique"))
)

print("중분류×시나리오별 격자 수 최솟값:", scenario_check["격자수"].min())
print("중분류×시나리오별 격자 수 최댓값:", scenario_check["격자수"].max())
display(scenario_check.head(10))

## 4. 접근성 점수 변화 요약
- 시나리오별 접근성 평균, 중앙값, 최소·최대값을 계산
- 문화누리대상자 추정 인구수를 가중치로 둔 가중평균 접근성을 함께 계산
- 점수 크기는 모형별 단위 차이가 있으므로 보조적으로 해석

In [ ]:
summary_list = []

group_cols = ["중분류", "거리감쇠방식", "문화누리대상자_수요가중치", "scenario"]

for keys, temp in h3_target.groupby(group_cols):
    category, decay_method, demand_weight, scenario = keys
    weight = temp["문화누리대상자_추정_인구수"].clip(lower=0)
    if weight.sum() > 0:
        weighted_mean = np.average(temp["접근성지수"], weights=weight)
    else:
        weighted_mean = np.nan

    summary_list.append({
        "중분류": category,
        "거리감쇠방식": decay_method,
        "거리감쇠방식_설명": DECAY_LABEL.get(decay_method, decay_method),
        "문화누리대상자_수요가중치": demand_weight,
        "scenario": scenario,
        "격자수": temp["GRID_CD"].nunique(),
        "접근가능격자수": (temp["접근가능_가맹점수"] > 0).sum(),
        "무접근격자수": (temp["접근가능_가맹점수"] <= 0).sum(),
        "접근성평균": temp["접근성지수"].mean(),
        "접근성중앙값": temp["접근성지수"].median(),
        "접근성최솟값": temp["접근성지수"].min(),
        "접근성최댓값": temp["접근성지수"].max(),
        "대상자_가중평균접근성": weighted_mean,
    })

scenario_summary = pd.DataFrame(summary_list)
scenario_summary = scenario_summary.sort_values(["중분류", "거리감쇠방식", "문화누리대상자_수요가중치"]).reset_index(drop=True)

print("시나리오 요약표 구조:", scenario_summary.shape)
display(scenario_summary.head(12))

scenario_summary.to_csv(
    SENSITIVITY_PATH / "h3sfca_sensitivity_시나리오_요약.csv",
    index=False,
    encoding="utf-8-sig"
)

## 5. 시나리오 간 순위상관 분석
- 같은 중분류 안에서 시나리오별 접근성 순위가 얼마나 유사한지 Spearman 상관으로 계산
- 값이 1에 가까울수록 시나리오를 바꿔도 지역 순위가 안정적
- 값이 낮을수록 거리감쇠 또는 수요가중치 설정에 따라 지역 순위가 흔들림

In [ ]:
spearman_list = []

for category, temp in h3_target.groupby("중분류"):
    wide = temp.pivot_table(
        index="GRID_CD",
        columns="scenario",
        values="접근성지수",
        aggfunc="first"
    )

    scenario_cols = sorted(wide.columns)

    for scenario_a in scenario_cols:
        for scenario_b in scenario_cols:
            series_a = wide[scenario_a]
            series_b = wide[scenario_b]

            if series_a.nunique(dropna=True) <= 1 or series_b.nunique(dropna=True) <= 1:
                spearman = 1.0 if series_a.equals(series_b) else np.nan
            else:
                spearman = series_a.corr(series_b, method="spearman")

            spearman_list.append({
                "중분류": category,
                "scenario_a": scenario_a,
                "scenario_b": scenario_b,
                "spearman": spearman
            })

spearman_matrix = pd.DataFrame(spearman_list)

spearman_offdiag = spearman_matrix[spearman_matrix["scenario_a"] != spearman_matrix["scenario_b"]].copy()
spearman_summary = (
    spearman_offdiag
    .groupby("중분류", as_index=False)["spearman"]
    .agg(["mean", "min", "max"])
    .reset_index()
    .rename(columns={"mean": "평균순위상관", "min": "최소순위상관", "max": "최대순위상관"})
)

print("순위상관 행렬 구조:", spearman_matrix.shape)
print("전체 평균 순위상관:", round(spearman_offdiag["spearman"].mean(), 4))
print("전체 최소 순위상관:", round(spearman_offdiag["spearman"].min(), 4))
display(spearman_summary.sort_values("평균순위상관"))

spearman_matrix.to_csv(
    SENSITIVITY_PATH / "h3sfca_sensitivity_spearman_순위상관.csv",
    index=False,
    encoding="utf-8-sig"
)

## 6. 취약지역 중복률 분석
- 중분류·시나리오별 접근성 하위 20% 격자를 취약지역으로 설정
- 시나리오 쌍별 취약지역 중복률을 Jaccard 지수로 계산

\[
J(A, B) = \frac{|A \cap B|}{|A \cup B|}
\]

- 값이 1에 가까울수록 취약지역 선정 결과가 안정적
- 값이 낮을수록 설정 변화에 따라 취약지역 판정이 달라짐

In [ ]:
vulnerable_dict = {}
vulnerable_count_list = []

for keys, temp in h3_target.groupby(["중분류", "scenario"]):
    category, scenario = keys
    temp_sort = temp.sort_values(["접근성지수", "GRID_CD"]).reset_index(drop=True)
    vulnerable_n = int(np.ceil(len(temp_sort) * 0.2))
    vulnerable_grid = set(temp_sort.head(vulnerable_n)["GRID_CD"])

    vulnerable_dict[(category, scenario)] = vulnerable_grid
    vulnerable_count_list.append({
        "중분류": category,
        "scenario": scenario,
        "취약격자수": len(vulnerable_grid),
        "취약판정대상격자수": len(temp_sort)
    })

jaccard_list = []

for category in sorted(h3_target["중분류"].unique()):
    scenario_cols = sorted(h3_target.loc[h3_target["중분류"] == category, "scenario"].unique())

    for scenario_a in scenario_cols:
        for scenario_b in scenario_cols:
            set_a = vulnerable_dict[(category, scenario_a)]
            set_b = vulnerable_dict[(category, scenario_b)]
            union_n = len(set_a | set_b)
            inter_n = len(set_a & set_b)
            jaccard = inter_n / union_n if union_n > 0 else np.nan

            jaccard_list.append({
                "중분류": category,
                "scenario_a": scenario_a,
                "scenario_b": scenario_b,
                "교집합격자수": inter_n,
                "합집합격자수": union_n,
                "jaccard": jaccard
            })

vulnerable_count = pd.DataFrame(vulnerable_count_list)
jaccard_matrix = pd.DataFrame(jaccard_list)

jaccard_offdiag = jaccard_matrix[jaccard_matrix["scenario_a"] != jaccard_matrix["scenario_b"]].copy()
jaccard_summary = (
    jaccard_offdiag
    .groupby("중분류", as_index=False)["jaccard"]
    .agg(["mean", "min", "max"])
    .reset_index()
    .rename(columns={"mean": "평균중복률", "min": "최소중복률", "max": "최대중복률"})
)

print("취약지역 중복률 행렬 구조:", jaccard_matrix.shape)
print("전체 평균 중복률:", round(jaccard_offdiag["jaccard"].mean(), 4))
print("전체 최소 중복률:", round(jaccard_offdiag["jaccard"].min(), 4))
display(jaccard_summary.sort_values("평균중복률"))
display(vulnerable_count.head(12))

jaccard_matrix.to_csv(
    SENSITIVITY_PATH / "h3sfca_sensitivity_jaccard_취약지역중복률.csv",
    index=False,
    encoding="utf-8-sig"
)

## 7. 주요 결과 요약
- 순위상관이 낮은 중분류는 설정 변화에 따라 지역 순위가 크게 바뀐 중분류
- 취약지역 중복률이 낮은 중분류는 하위 20% 취약지역 선정이 민감한 중분류
- 음악·체육용품은 ML 선호확률이 없어 선호 미반영 SFCA 값으로 대체한 분류이므로 민감도 비교 결과에서 제외함

In [ ]:
low_spearman = spearman_summary.sort_values("평균순위상관").head(5)
low_jaccard = jaccard_summary.sort_values("평균중복률").head(5)

print("순위상관이 낮은 중분류 TOP 5")
display(low_spearman)

print("취약지역 중복률이 낮은 중분류 TOP 5")
display(low_jaccard)

print("저장된 산출물")
for path in sorted(SENSITIVITY_PATH.glob("*.csv")):
    print("-", path.name)

## 8. 민감 분류의 설정 요인별 영향 분해
- 대상 중분류: 공연, 관광지, 미술
- 수요가중치 민감도: 거리감쇠 방식을 고정하고 λ 값만 변경하여 순위 변동을 계산
- 거리감쇠 민감도: λ 값을 고정하고 거리감쇠 방식만 변경하여 순위 변동을 계산
- 변동값은 격자별 취약백분위의 평균 절대 차이로 계산

In [ ]:
SENSITIVE_CATEGORIES = ["공연", "관광지", "미술"]

h3_sensitive = h3_target[h3_target["중분류"].isin(SENSITIVE_CATEGORIES)].copy()
h3_sensitive = h3_sensitive.sort_values(["중분류", "scenario", "접근성지수", "GRID_CD"]).copy()
h3_sensitive["접근성순위백분위"] = (
    h3_sensitive
    .groupby(["중분류", "scenario"], observed=True)["접근성지수"]
    .rank(method="first", pct=True, ascending=True)
)
h3_sensitive["취약백분위"] = 1 - h3_sensitive["접근성순위백분위"]

factor_effect = pd.read_csv(SENSITIVITY_PATH / "h3sfca_sensitivity_민감분류_요인별_순위변동.csv")

factor_summary = (
    factor_effect
    .groupby(["중분류", "민감도요인"], as_index=False)
    .agg(
        평균_취약백분위변동=("평균_취약백분위변동", "mean"),
        최대_취약백분위변동=("최대_취약백분위변동", "max"),
        평균_spearman=("spearman", "mean"),
        최소_spearman=("spearman", "min")
    )
    .sort_values(["중분류", "평균_취약백분위변동"], ascending=[True, False])
)

print("민감도 요인별 상세표 저장:", SENSITIVITY_PATH / "h3sfca_sensitivity_민감분류_요인별_순위변동.csv")
display(factor_summary)
display(factor_effect.sort_values("평균_취약백분위변동", ascending=False).head(12))

## 9. 민감 분류의 격자별 공간 민감도
- 격자별 평균 취약백분위, 취약백분위 범위, 취약 선정 횟수를 계산
- 수요가중치 변화에 따른 민감도와 거리감쇠 변화에 따른 민감도를 분리
- 값이 클수록 해당 격자의 취약성 판단이 시나리오 설정에 민감함

In [ ]:
grid_sensitivity = pd.read_csv(SENSITIVITY_PATH / "h3sfca_sensitivity_민감분류_격자별_민감도.csv")

print("격자별 민감도 테이블 저장:", SENSITIVITY_PATH / "h3sfca_sensitivity_민감분류_격자별_민감도.csv")
print("데이터 구조:", grid_sensitivity.shape)

sensitivity_by_gu = (
    grid_sensitivity
    .groupby(["중분류", "시군구"], as_index=False)
    .agg(
        평균취약백분위=("평균취약백분위", "mean"),
        평균취약백분위범위=("취약백분위범위", "mean"),
        평균수요가중치민감도=("수요가중치민감도", "mean"),
        평균거리감쇠민감도=("거리감쇠민감도", "mean"),
        취약선정횟수합=("취약선정횟수", "sum"),
        문화누리대상자수=("문화누리대상자_추정_인구수", "sum")
    )
)

print("공간 민감도가 높은 시군구 TOP 15")
display(sensitivity_by_gu.sort_values("평균취약백분위범위", ascending=False).head(15))

print("격자 단위 민감도 TOP 20")
display(grid_sensitivity.sort_values("취약백분위범위", ascending=False).head(20))

## 10. 2025 이용건수 기반 시나리오 회귀 비교
- 목적: 9개 H3SFCA 시나리오 중 어떤 거리감쇠·수요가중치 설정이 2025년 구별·중분류별 이용건수 예측오차를 가장 줄이는지 비교
- 종속변수: `log(1 + 이용건수)`
- 핵심 비교변수: 시나리오별 `log(1 + H3SFCA 접근성)`
- 공통 통제변수: 구별 문화누리대상자 추정 인구, 구별 총 추정 인구, 문화누리대상자 성연령 비율, 중분류 고정효과
- 제외 변수: 가맹점 수, 공급량, 접근 가능 가맹점 수는 접근성 지표 안에 이미 반영되어 제외

In [ ]:
regression_result = pd.read_csv(SENSITIVITY_PATH / "h3sfca_usage_regression_2025_scenario_performance.csv")
scenario_corr = pd.read_csv(SENSITIVITY_PATH / "h3sfca_usage_correlation_2025_scenario_category.csv")
model_data = pd.read_csv(SENSITIVITY_PATH / "h3sfca_usage_regression_2025_model_data.csv")

print("회귀 비교 데이터:", model_data.shape)
print("시나리오 성능표:", regression_result.shape)
display(regression_result.sort_values("log_RMSE"))
print("중분류별 상관 일부")
display(scenario_corr.head(20))

### 회귀 비교 주요 결과
- 가장 낮은 log RMSE 시나리오: `piecewise_lambda_1.5`
- 거리감쇠 방식: `piecewise`
- 문화누리대상자 수요가중치: `1.5`
- log RMSE: `0.3123`
- log MAE: `0.2087`
- Adj. R²: `0.9751`
- 비교 목적은 변수 중요도 해석이 아니라 동일 회귀 조건에서 접근성 시나리오별 예측오차를 비교하는 것임.

## 7. 이용금액 기준 접근성 시나리오 검증
- 목적: 이용건수 기준 검증과 동일한 방식으로 2024년·2025년 이용금액을 비교함.
- 기준: 구별·중분류별 이용금액과 구별·중분류별 H3SFCA 접근성을 결합함.
- 해석: 2024년은 현재 접근성 산출값과 결합한 보조 검토로 해석함.


### 7-1. 분석식
- 종속변수: 중분류별 문화누리카드 이용금액
- 변환식: $\log(1+\text{이용금액})$
- 비교대상: 거리감쇠 3개 방식 × 문화누리 수요가중치 3개 조합

$$
\log(1+Y_{g,c})
= \beta_0
+ \beta_1 \log(1+A_{g,c,s})
+ \beta_2 \log(1+P^{MNC}_g)
+ \beta_3 \log(1+P^{Total}_g)
+ \mathbf{\gamma X_g}
+ \delta_c
+ \epsilon_{g,c}
$$

- $Y_{g,c}$: 구 $g$, 중분류 $c$의 이용금액
- $A_{g,c,s}$: 시나리오 $s$의 H3SFCA 접근성
- $P^{MNC}_g$: 문화누리대상자 추정인구
- $P^{Total}_g$: 총 추정인구
- $\mathbf{X_g}$: 성연령 구성비
- $\delta_c$: 중분류 고정효과


In [ ]:
# 이용금액 기준 민감도 검증
import pathlib
import numpy as np
import pandas as pd

BASE_PATH = pathlib.Path().resolve()

if BASE_PATH.name == "access":
    BASE_PATH = BASE_PATH.parents[1]
elif BASE_PATH.name == "notebooks":
    BASE_PATH = BASE_PATH.parent
elif BASE_PATH.name != "oracle_mnc_project" and (BASE_PATH / "oracle_mnc_project").exists():
    BASE_PATH = BASE_PATH / "oracle_mnc_project"

RAW_USAGE_PATH = BASE_PATH / "data" / "raw" / "mnc_card" / "mnc_seoul_usage_issuance_2021_2025.xlsx"
SENSITIVITY_PATH = BASE_PATH / "notebooks" / "access" / "OUTPUT" / "h3sfca_sensitivity"

MODEL_DATA_PATH = SENSITIVITY_PATH / "h3sfca_usage_regression_2025_model_data.csv"
PROCESSED_2025_PATH = BASE_PATH / "data" / "processed" / "table_design" / "usage_access_model_ready_2025.csv"

AMOUNT_PERFORMANCE_PATH = SENSITIVITY_PATH / "h3sfca_usage_amount_regression_2024_2025_scenario_performance.csv"
AMOUNT_CORRELATION_PATH = SENSITIVITY_PATH / "h3sfca_usage_amount_correlation_2024_2025_scenario_category.csv"
AMOUNT_COEF_PATH = SENSITIVITY_PATH / "h3sfca_usage_amount_regression_2024_2025_coefficients.csv"

SENSITIVITY_PATH.mkdir(parents=True, exist_ok=True)

print("원자료:", RAW_USAGE_PATH)
print("접근성 검증 테이블:", MODEL_DATA_PATH)
print("산출 폴더:", SENSITIVITY_PATH)


# 1. 이용금액 데이터 불러오기
amount_mapping = {
    "공연": ["공연\n(원)"],
    "관광지": ["관광명소\n(원)", "휴양림/캠핑장\n(원)", "동ㆍ식물원\n(원)", "온천\n(원)", "체험관광\n(원)", "테마파크\n(원)"],
    "도서": ["도서\n(원)"],
    "문화체험": ["문화체험\n(원)", "직업체험\n(원)", "문화일반\n(원)"],
    "미술": ["전시\n(원)", "공예\n(원)", "사진관\n(원)"],
    "스포츠관람": ["스포츠관람\n(원)"],
    "영상": ["영화\n(원)"],
    "체육시설": ["체육시설\n(원)"],
}

amount_list = []

for year in [2024, 2025]:
    usage_raw = pd.read_excel(RAW_USAGE_PATH, sheet_name=str(year))
    usage_raw["광역"] = usage_raw["광역"].astype(str).str.strip()
    usage_seoul = usage_raw[usage_raw["광역"] == "서울"].copy()
    
    print(f"\n{year}년 서울 구 개수:", usage_seoul["기초"].nunique())
    print(f"{year}년 서울 데이터 구조:", usage_seoul.shape)
    
    for category, cols in amount_mapping.items():
        missing_cols = [col for col in cols if col not in usage_seoul.columns]
        if missing_cols:
            raise KeyError(f"{year}년 {category} 이용금액 칼럼 없음: {missing_cols}")
        
        temp = usage_seoul[["기초"] + cols].copy()
        temp["year"] = year
        temp["시군구"] = temp["기초"].astype(str).str.strip()
        temp["중분류"] = category
        temp["이용금액"] = temp[cols].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)
        
        amount_list.append(temp[["year", "시군구", "중분류", "이용금액"]])

usage_amount = pd.concat(amount_list, ignore_index=True)

print("\n이용금액 long 데이터 구조:", usage_amount.shape)
print("연도:", sorted(usage_amount["year"].unique()))
print("연도별 구 개수")
print(usage_amount.groupby("year")["시군구"].nunique())
print("연도별 중분류 개수")
print(usage_amount.groupby("year")["중분류"].nunique())
print("이용금액 결측:", usage_amount["이용금액"].isna().sum())
print("이용금액 0 이하:", (usage_amount["이용금액"] <= 0).sum())

display(
    usage_amount
    .groupby(["year", "중분류"], as_index=False)["이용금액"]
    .sum()
    .sort_values(["year", "이용금액"], ascending=[True, False])
)


# 2. 2025년 기존 가공 이용금액과 매핑 결과 일치 여부 확인
if PROCESSED_2025_PATH.exists():
    processed_2025 = pd.read_csv(PROCESSED_2025_PATH, encoding="utf-8-sig")
    compare_2025 = processed_2025[["시군구", "중분류", "use_amount"]].merge(
        usage_amount[usage_amount["year"] == 2025],
        on=["시군구", "중분류"],
        how="outer"
    )
    compare_2025["차이"] = compare_2025["use_amount"].fillna(0) - compare_2025["이용금액"].fillna(0)
    
    print("\n2025년 기존 가공테이블과 이용금액 매핑 비교")
    print("비교 데이터 구조:", compare_2025.shape)
    print("최대 절대 차이:", compare_2025["차이"].abs().max())
else:
    print("\n2025년 기존 가공테이블 없음: 매핑 검증 생략")


# 3. 접근성 시나리오 테이블 결합
model_base = pd.read_csv(MODEL_DATA_PATH, encoding="utf-8-sig")
model_base = model_base.drop(columns=["year", "use_count", "log_use_count"], errors="ignore")

model_amount = model_base.merge(
    usage_amount,
    on=["시군구", "중분류"],
    how="left"
)

model_amount["log_use_amount"] = np.log1p(model_amount["이용금액"])
model_amount["log_access"] = np.log1p(model_amount["accessibility"])
model_amount["log_mnc_pop"] = np.log1p(model_amount["mnc_pop"])
model_amount["log_total_pop"] = np.log1p(model_amount["total_pop"])

print("\n회귀 분석용 데이터 구조:", model_amount.shape)
print("접근성 결측:", model_amount["accessibility"].isna().sum())
print("이용금액 결측:", model_amount["이용금액"].isna().sum())
print("시나리오 수:", model_amount["scenario"].nunique())


# 4. 시나리오별 회귀 및 예측오차
feature_cols = [
    "log_access",
    "log_mnc_pop",
    "log_total_pop",
    "ratio_mnc_age_6_19",
    "ratio_mnc_age_20_39",
    "ratio_mnc_age_40_59",
    "ratio_mnc_female",
]

scenario_results = []
coef_results = []

for year in sorted(model_amount["year"].unique()):
    year_data = model_amount[model_amount["year"] == year].copy()
    
    for scenario in sorted(year_data["scenario"].unique()):
        temp = year_data[year_data["scenario"] == scenario].copy()
        
        X = temp[feature_cols].copy()
        category_dummies = pd.get_dummies(temp["중분류"], prefix="중분류", drop_first=True, dtype=float)
        X = pd.concat([X, category_dummies], axis=1)
        X = X.fillna(0)
        X.insert(0, "intercept", 1.0)
        
        y = temp["log_use_amount"].to_numpy()
        beta, *_ = np.linalg.lstsq(X.to_numpy(), y, rcond=None)
        pred_log = X.to_numpy() @ beta
        pred_amount = np.expm1(pred_log)
        
        resid_log = y - pred_log
        resid_amount = temp["이용금액"].to_numpy() - pred_amount
        n = len(temp)
        p = X.shape[1] - 1
        
        sse = np.sum(resid_log ** 2)
        sst = np.sum((y - y.mean()) ** 2)
        r2 = 1 - sse / sst if sst > 0 else np.nan
        adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1) if n > p + 1 else np.nan
        
        scenario_results.append({
            "year": year,
            "scenario": scenario,
            "decay_type": scenario.split("_lambda_")[0],
            "mnc_weight": float(scenario.split("_lambda_")[1]),
            "n": n,
            "log_amount_rmse": np.sqrt(np.mean(resid_log ** 2)),
            "log_amount_mae": np.mean(np.abs(resid_log)),
            "amount_rmse": np.sqrt(np.mean(resid_amount ** 2)),
            "amount_mae": np.mean(np.abs(resid_amount)),
            "adj_r2": adj_r2,
            "access_coef": beta[list(X.columns).index("log_access")],
        })
        
        for col, value in zip(X.columns, beta):
            coef_results.append({
                "year": year,
                "scenario": scenario,
                "feature": col,
                "coef": value,
            })

scenario_perf = pd.DataFrame(scenario_results)
coef_table = pd.DataFrame(coef_results)

scenario_perf.to_csv(AMOUNT_PERFORMANCE_PATH, index=False, encoding="utf-8-sig")
coef_table.to_csv(AMOUNT_COEF_PATH, index=False, encoding="utf-8-sig")

print("\n이용금액 기준 시나리오 성능: log RMSE 낮은 순")
display(
    scenario_perf
    .sort_values(["year", "log_amount_rmse"])
    .groupby("year")
    .head(9)
)

print("\n이용금액 기준 시나리오 성능: 원금액 RMSE 낮은 순")
display(
    scenario_perf
    .sort_values(["year", "amount_rmse"])
    .groupby("year")
    .head(9)
)

print("\n접근성 계수")
display(
    scenario_perf[["year", "scenario", "access_coef"]]
    .sort_values(["year", "scenario"])
)


# 5. 중분류별 상관관계
correlation_results = []

for year in sorted(model_amount["year"].unique()):
    year_data = model_amount[model_amount["year"] == year].copy()
    
    for scenario in sorted(year_data["scenario"].unique()):
        temp_s = year_data[year_data["scenario"] == scenario].copy()
        
        for category in sorted(temp_s["중분류"].unique()):
            temp_c = temp_s[temp_s["중분류"] == category].copy()
            
            pearson = temp_c["accessibility"].corr(temp_c["이용금액"], method="pearson")
            spearman = temp_c["accessibility"].corr(temp_c["이용금액"], method="spearman")
            
            correlation_results.append({
                "year": year,
                "scenario": scenario,
                "중분류": category,
                "pearson_corr": pearson,
                "spearman_corr": spearman,
            })

amount_corr = pd.DataFrame(correlation_results)
amount_corr.to_csv(AMOUNT_CORRELATION_PATH, index=False, encoding="utf-8-sig")

print("\n중분류별 이용금액-접근성 상관관계 평균")
display(
    amount_corr
    .groupby(["year", "중분류"], as_index=False)[["pearson_corr", "spearman_corr"]]
    .mean()
    .sort_values(["year", "pearson_corr"], ascending=[True, False])
)

print("\n저장 완료")
print(AMOUNT_PERFORMANCE_PATH)
print(AMOUNT_CORRELATION_PATH)
print(AMOUNT_COEF_PATH)


### 7-2. 주요 결과
- 이용금액 기준에서는 2024년·2025년 모두 로그 RMSE 기준으로 거리감쇠 없음 시나리오가 근소하게 우수함.
- 원금액 RMSE 기준에서는 2024년·2025년 모두 Gaussian 감쇠가 근소하게 우수함.
- 접근성 계수는 모든 시나리오에서 음수로 나타남.
- 체육시설은 이용금액과 접근성의 상관이 높게 나타났지만, 공연·도서·미술·영상은 음의 상관 또는 약한 상관을 보임.
- 따라서 이용금액 기준 검증은 H3SFCA 접근성이 전체 이용금액을 안정적으로 설명한다고 보기 어려운 결과임.

#### 2024년 로그 RMSE 기준 상위 시나리오
|순위|시나리오|log RMSE|log MAE|Adj. R²|접근성 계수|
|---:|---|---:|---:|---:|---:|
|1|none_lambda_1.0|0.260712|0.179553|0.982270|-9.378042|
|2|none_lambda_1.2|0.260712|0.179553|0.982270|-9.471197|
|3|none_lambda_1.5|0.260713|0.179553|0.982270|-9.609929|

#### 2025년 로그 RMSE 기준 상위 시나리오
|순위|시나리오|log RMSE|log MAE|Adj. R²|접근성 계수|
|---:|---|---:|---:|---:|---:|
|1|none_lambda_1.5|0.270912|0.183533|0.979893|-3.075232|
|2|none_lambda_1.2|0.270913|0.183533|0.979893|-3.028712|
|3|none_lambda_1.0|0.270913|0.183533|0.979893|-2.997031|

#### 해석상 주의
- Adj. R²가 높은 이유는 중분류 고정효과가 이용금액 규모 차이를 크게 설명했기 때문임.
- 접근성만으로 이용금액을 설명했다는 의미가 아님.
- 2024년 이용금액 검증은 접근성 산출값이 2025년 기준이므로 연도 정합성이 완전하지 않음.


## 11. PPT용 시각화 코드

- 9개 실험안을 최종 적용안(`piecewise_lambda_1.2`)과 비교한다.
- 표는 파일로 저장하지 않고 `display()`로만 확인한다.
- 그래프만 발표용 이미지로 저장하고 노트북에도 바로 표시한다.
- Noto Sans KR variable font가 Thin으로 잡히는 문제를 피하기 위해 Medium/Bold 정적 인스턴스를 직접 등록한다.
- 유의성 검정은 소표본/비정규 가능성을 고려해 비모수 검정을 사용한다.

In [ ]:
# PPT_FIGURE_MARKER_5장: 9개 실험안별 요약 및 유의성 검정
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects
from matplotlib import font_manager
from IPython.display import display, Markdown
from scipy.stats import mannwhitneyu, wilcoxon

BASELINE_SCENARIO = "piecewise_lambda_1.2"
SCENARIO_ORDER = [
    "piecewise_lambda_1.0", "piecewise_lambda_1.2", "piecewise_lambda_1.5",
    "gaussian_lambda_1.0", "gaussian_lambda_1.2", "gaussian_lambda_1.5",
    "none_lambda_1.0", "none_lambda_1.2", "none_lambda_1.5",
]
DECAY_LABEL = {"piecewise": "구간형", "gaussian": "가우시안", "none": "무감쇠"}


def find_project_base():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "notebooks" / "access" / "OUTPUT" / "h3sfca_sensitivity").exists():
            return candidate
    return cwd


BASE = find_project_base()
DATA_DIR = BASE / "notebooks" / "access" / "OUTPUT" / "h3sfca_sensitivity"
IMAGE_DIR = BASE / "notebooks" / "access" / "IMAGE" / "sensitivity_chapter"
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

BG = "#FBF6EF"
TEXT = "#2A211D"
MUTED = "#746C66"
AXIS = "#D8D1CB"
GREY = "#C9C3BE"
ORANGE = "#F46B2F"

NOTO_VF_PATH = Path("C:/Windows/Fonts/NotoSansKR-VF.ttf")
STATIC_FONT_DIR = IMAGE_DIR / "_fonts"
MEDIUM_FONT_PATH = STATIC_FONT_DIR / "NotoSansKR-Medium.ttf"
BOLD_FONT_PATH = STATIC_FONT_DIR / "NotoSansKR-Bold.ttf"


def ensure_static_noto_fonts():
    if MEDIUM_FONT_PATH.exists() and BOLD_FONT_PATH.exists():
        return
    if not NOTO_VF_PATH.exists():
        return
    STATIC_FONT_DIR.mkdir(parents=True, exist_ok=True)
    from fontTools.ttLib import TTFont
    from fontTools.varLib import instancer
    for out_path, weight in [(MEDIUM_FONT_PATH, 500), (BOLD_FONT_PATH, 700)]:
        if out_path.exists():
            continue
        font = TTFont(str(NOTO_VF_PATH))
        static_font = instancer.instantiateVariableFont(font, {"wght": weight}, inplace=False)
        static_font.save(str(out_path))


ensure_static_noto_fonts()

if MEDIUM_FONT_PATH.exists() and BOLD_FONT_PATH.exists():
    font_manager.fontManager.addfont(str(MEDIUM_FONT_PATH))
    font_manager.fontManager.addfont(str(BOLD_FONT_PATH))
    BODY_FONT = font_manager.FontProperties(fname=str(MEDIUM_FONT_PATH))
    TITLE_FONT = font_manager.FontProperties(fname=str(BOLD_FONT_PATH))
elif NOTO_VF_PATH.exists():
    font_manager.fontManager.addfont(str(NOTO_VF_PATH))
    BODY_FONT = font_manager.FontProperties(fname=str(NOTO_VF_PATH))
    TITLE_FONT = font_manager.FontProperties(fname=str(NOTO_VF_PATH))
else:
    BODY_FONT = font_manager.FontProperties(family="Malgun Gothic")
    TITLE_FONT = font_manager.FontProperties(family="Malgun Gothic", weight="bold")

plt.rcParams.update({
    "font.family": BODY_FONT.get_name(),
    "axes.unicode_minus": False,
    "figure.facecolor": BG,
    "axes.facecolor": BG,
    "savefig.facecolor": BG,
    "text.color": TEXT,
    "axes.labelcolor": TEXT,
    "xtick.color": MUTED,
    "ytick.color": TEXT,
})


def parse_scenarios(df):
    result = df.copy()
    for col in ["scenario_a", "scenario_b"]:
        result[f"{col}_decay"] = result[col].str.split("_lambda_").str[0]
        result[f"{col}_lambda"] = result[col].str.split("_lambda_").str[1].astype(float)
    return result


def scenario_label(scenario):
    decay, lambda_value = scenario.split("_lambda_")
    label = f"{DECAY_LABEL[decay]} lambda {lambda_value}"
    if scenario == BASELINE_SCENARIO:
        label += " (최종)"
    return label


def wilcoxon_less_than_one(values):
    values = np.asarray(values, dtype=float)
    diff = values - 1.0
    if np.allclose(diff, 0, atol=1e-12):
        return 1.0
    return float(wilcoxon(diff, alternative="less", zero_method="wilcox").pvalue)


def significance_label(p_value, alpha=0.05):
    if pd.isna(p_value):
        return "검정불가"
    return "유의" if p_value < alpha else "비유의"


def apply_font_to_axis(ax):
    for text in ax.get_xticklabels() + ax.get_yticklabels():
        text.set_fontproperties(BODY_FONT)
        text.set_color(MUTED if text in ax.get_xticklabels() else TEXT)


def setup_axis(ax):
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(AXIS)
    ax.spines["bottom"].set_color(AXIS)
    ax.tick_params(axis="both", labelsize=13, width=1.0, color=MUTED)
    apply_font_to_axis(ax)


def add_suptitle(fig, title):
    title_obj = fig.suptitle(title, y=0.958, fontsize=18, fontproperties=TITLE_FONT, color=TEXT)
    title_obj.set_path_effects([])


def add_title(ax, title):
    title_obj = ax.set_title(title, fontsize=20, fontproperties=TITLE_FONT, color=TEXT, pad=16)
    title_obj.set_path_effects([])


def add_legend(fig):
    from matplotlib.lines import Line2D
    handles = [
        Line2D([0], [0], marker="o", color="none", markerfacecolor=ORANGE, markeredgecolor=ORANGE, markersize=11),
        Line2D([0], [0], marker="o", color="none", markerfacecolor=GREY, markeredgecolor=GREY, markersize=11),
    ]
    legend = fig.legend(
        handles,
        ["최종 적용안", "비교 실험안"],
        loc="upper center",
        bbox_to_anchor=(0.5, 0.890),
        ncol=2,
        frameon=False,
        prop=BODY_FONT,
        fontsize=14,
        handlelength=1.8,
        columnspacing=2.4,
    )
    for text in legend.get_texts():
        text.set_color(TEXT)
        text.set_fontproperties(BODY_FONT)


spearman_raw = pd.read_csv(DATA_DIR / "h3sfca_sensitivity_spearman_순위상관.csv", encoding="utf-8-sig")
jaccard_raw = pd.read_csv(DATA_DIR / "h3sfca_sensitivity_jaccard_취약지역중복률.csv", encoding="utf-8-sig")
factor_raw = pd.read_csv(DATA_DIR / "h3sfca_sensitivity_민감분류_요인별_순위변동.csv", encoding="utf-8-sig")

spearman = parse_scenarios(spearman_raw)
jaccard = parse_scenarios(jaccard_raw)

rows = []
for scenario in SCENARIO_ORDER:
    decay, lambda_value = scenario.split("_lambda_")
    jaccard_values = jaccard[(jaccard["scenario_a"] == BASELINE_SCENARIO) & (jaccard["scenario_b"] == scenario)]["jaccard"].to_numpy()
    spearman_values = spearman[(spearman["scenario_a"] == BASELINE_SCENARIO) & (spearman["scenario_b"] == scenario)]["spearman"].to_numpy()

    jaccard_p = wilcoxon_less_than_one(jaccard_values)
    spearman_p = wilcoxon_less_than_one(spearman_values)
    rows.append({
        "실험안": scenario_label(scenario),
        "거리감쇠": DECAY_LABEL[decay],
        "수요가중치": float(lambda_value),
        "평균_Jaccard": jaccard_values.mean(),
        "최소_Jaccard": jaccard_values.min(),
        "Jaccard_p": jaccard_p,
        "Jaccard_검정": significance_label(jaccard_p),
        "평균_Spearman": spearman_values.mean(),
        "최소_Spearman": spearman_values.min(),
        "Spearman_p": spearman_p,
        "Spearman_검정": significance_label(spearman_p),
    })

scenario_summary = pd.DataFrame(rows)

factor_test_rows = []
for metric, label in [
    ("평균_취약백분위변동", "취약백분위 변동"),
    ("spearman", "순위 상관계수"),
]:
    distance_values = factor_raw[factor_raw["민감도요인"] == "거리감쇠"][metric]
    demand_values = factor_raw[factor_raw["민감도요인"] == "수요가중치"][metric]
    p_value = float(mannwhitneyu(distance_values, demand_values, alternative="two-sided").pvalue)
    factor_test_rows.append({
        "검정대상": label,
        "검정방법": "Mann-Whitney U",
        "거리감쇠_평균": distance_values.mean(),
        "수요가중치_평균": demand_values.mean(),
        "p_value": p_value,
        "해석": significance_label(p_value),
    })

factor_test_summary = pd.DataFrame(factor_test_rows)
factor_test_summary.loc[factor_test_summary["검정대상"] == "취약백분위 변동", ["거리감쇠_평균", "수요가중치_평균"]] *= 100
factor_test_summary.loc[factor_test_summary["검정대상"] == "취약백분위 변동", "검정대상"] = "취약백분위 변동(pp)"

display(Markdown(
    f"데이터 로드 완료: 9개 실험안, 최종 적용안 기준 `{BASELINE_SCENARIO}` / 그래프 폰트: Noto Sans KR Medium·Bold"
))

In [ ]:
# PPT_FIGURE_MARKER_5장: 9개 실험안별 요약표 display
scenario_display = scenario_summary.copy()
scenario_display["수요가중치"] = scenario_display["수요가중치"].map(lambda value: f"{value:.1f}")

display(Markdown("### 9개 실험안별 Jaccard 중복률 및 Spearman 순위 상관계수"))
display(
    scenario_display.style
    .format({
        "평균_Jaccard": "{:.4f}",
        "최소_Jaccard": "{:.4f}",
        "Jaccard_p": "{:.4f}",
        "평균_Spearman": "{:.5f}",
        "최소_Spearman": "{:.5f}",
        "Spearman_p": "{:.4f}",
    })
    .hide(axis="index")
)

display(Markdown("### 요인별 민감도 차이 유의성 검정"))
display(
    factor_test_summary.style
    .format({
        "거리감쇠_평균": "{:.4f}",
        "수요가중치_평균": "{:.4f}",
        "p_value": "{:.2e}",
    })
    .hide(axis="index")
)

In [ ]:
# PPT_FIGURE_MARKER_5장: 실험안별 Jaccard 중복률 및 Spearman 순위 상관계수
# 축은 0~1 전체 범위로 고정해 차이를 과장하지 않고, 막대는 명확히 보이도록 표시한다.
plot_df = scenario_summary.copy()
colors = [ORANGE if row == BASELINE_SCENARIO else GREY for row in SCENARIO_ORDER]

fig, axes = plt.subplots(1, 2, figsize=(16.0, 7.4), sharey=True)
fig.subplots_adjust(top=0.805, left=0.22, right=0.95, bottom=0.12, wspace=0.22)
add_suptitle(fig, "실험안별 Jaccard 중복률 및 Spearman 순위 상관계수")

legend_handles = [
    plt.Rectangle((0, 0), 1, 1, color=ORANGE),
    plt.Rectangle((0, 0), 1, 1, color=GREY),
]
legend = fig.legend(
    legend_handles,
    ["최종 적용안", "비교 실험안"],
    loc="upper center",
    bbox_to_anchor=(0.5, 0.890),
    ncol=2,
    frameon=False,
    prop=BODY_FONT,
    fontsize=13,
    handlelength=1.9,
    columnspacing=2.4,
)
for text in legend.get_texts():
    text.set_color(TEXT)
    text.set_fontproperties(BODY_FONT)

metrics = [
    (axes[0], "평균_Jaccard", "Jaccard 중복률", "{:.3f}"),
    (axes[1], "평균_Spearman", "Spearman 순위 상관계수", "{:.5f}"),
]

y = np.arange(len(plot_df))
for ax, metric, title, formatter in metrics:
    # 0~1 전체 기준을 얇은 배경 막대로 먼저 깔고 실제 값을 그 위에 올린다.
    ax.barh(y, 1.0, color="#E6DED7", height=0.50, zorder=1)
    bars = ax.barh(y, plot_df[metric], color=colors, height=0.50, zorder=2)
    ax.axvline(1.0, color=AXIS, linewidth=1.0, zorder=3)
    ax.set_xlim(0.0, 1.04)
    ax.set_xticks([0.0, 0.5, 1.0])
    ax.set_yticks(y)
    ax.set_yticklabels(plot_df["실험안"], fontsize=14, fontproperties=BODY_FONT)
    setup_axis(ax)
    add_title(ax, title)
    ax.tick_params(axis="y", length=0, pad=10)

    for bar, value in zip(bars, plot_df[metric]):
        ax.text(
            min(value + 0.012, 1.025),
            bar.get_y() + bar.get_height() / 2,
            formatter.format(value),
            va="center",
            ha="left",
            fontsize=11,
            color=TEXT,
            fontproperties=BODY_FONT,
        )

axes[0].invert_yaxis()
axes[1].tick_params(axis="y", labelleft=False)
axes[1].spines["left"].set_visible(False)

summary_image_path = IMAGE_DIR / "5장_실험안별_Jaccard_Spearman_요약.png"
fig.savefig(summary_image_path, dpi=240, bbox_inches="tight", pad_inches=0.16)
display(fig)
plt.close(fig)
display(Markdown(f"저장 경로: `{summary_image_path}`"))

In [ ]:
# PPT_FIGURE_MARKER_5장: 검정 결과 해석문 display
comparison_rows = scenario_summary[~scenario_summary["실험안"].str.contains("최종", regex=False)]
factor_movement = factor_test_summary[factor_test_summary["검정대상"] == "취약백분위 변동(pp)"].iloc[0]
factor_spearman = factor_test_summary[factor_test_summary["검정대상"] == "순위 상관계수"].iloc[0]

result_text = f"""
### 결과 요약

- 9개 실험안은 거리감쇠 3개(구간형, 가우시안, 무감쇠)와 수요가중치 3개(lambda 1.0, 1.2, 1.5)를 조합해 구성했다.
- 최종 적용안은 구간형 거리감쇠 + lambda 1.2다.
- 최종 적용안 대비 비교 실험안의 평균 Jaccard 중복률은 {comparison_rows['평균_Jaccard'].mean():.4f}, 평균 Spearman 순위 상관계수는 {comparison_rows['평균_Spearman'].mean():.5f}다.
- Wilcoxon signed-rank 검정 기준으로 Jaccard 중복률 감소는 유의하지 않았고, Spearman 순위 상관계수는 값의 감소 방향이 일관되어 유의하게 나타났다. 다만 Spearman 값 자체는 모두 0.990 이상으로 매우 높아 순위 안정성은 유지되었다.
- 요인별 민감도 차이는 Mann-Whitney U 검정 결과, 취약백분위 변동 기준 p={factor_movement['p_value']:.2e}, 순위 상관계수 기준 p={factor_spearman['p_value']:.2e}로 유의했다. 즉 수요가중치보다 거리감쇠 방식 변화가 통계적으로도 더 민감하게 작용했다.
"""

display(Markdown(result_text))